In [ ]:
BIBLIA_PATH = "../../data/processed/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.txt"
MAKE_VECTOR_STORE = True
MAKE_DATABASE = True

In [ ]:
from pprint import pprint
from bible_model import Verse

def get_verse(line: str) -> Verse:
    """
    Get metadata for the line.
    Line must be in the format:
    <book> <chapter>:<verse> <text>
    """
    book = " ".join(line.split(":")[0].split(" ")[:-1])
    chapter = line.split(":")[0].split(" ")[-1]
    verse = line.split(":")[1].split(" ")[0]
    text = " ".join(line.split(":")[1].split(" ")[1:])
    # return {"book": book, "chapter": chapter, "verse": verse, "text": text}
    return Verse(book, int(chapter), int(verse), text)


pprint(
    get_verse(
        "Gênesis 4:21 O nome do seu irmão era Jubal, que foi o pai de todos aqueles que tocam a cítara e os instrumentos de sopro."
    )
)

pprint(
    get_verse(
        "1 Samuel 1:1 Havia em Ramataim-Sofim um homem das montanhas de Efraim, chamado Elcana, filho de Jeroam, filho de Eliú, filho de Tou, filho de Suf, o efraimita."
    )
)

pprint(
    get_verse(
        "2 Reis 19:17 É verdade, Senhor, que os reis da Assíria destruíram as nações e devastaram os seus territórios,"
    )
)

In [ ]:
from typing import List
from langchain.schema import Document

# 1. Carrega o conteúdo da Bíblia
with open(BIBLIA_PATH, "r", encoding="utf-8") as f:
    biblia_texto = f.readlines()

# 2. Cada linha vira um documento individual
docs: List[Document] = []
for i, line in enumerate(biblia_texto):
    if len(line) > 0:
        verse = get_verse(line)
        docs.append(
            Document(
                page_content=verse.text,
                metadata={
                    "book": verse.book,
                    "chapter": verse.chapter,
                    "verse": verse.verse,
                    "line_number": i,
                },
            )
        )

print(docs[9999])

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

if MAKE_VECTOR_STORE:
    # 3. Inicializa o modelo de embedding
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    db = Chroma(
        collection_name="biblia",
        embedding_function=embedding_model,
        persist_directory="biblia_vectorstore",
    )
    # 4. Cria o índice vetorial
    ids = db.add_documents(docs)

In [ ]:
# Example query to search for similar documents
query = "No princípio, Deus criou o céu e a terra."
results = db.similarity_search(query, k=5)

# Print the results
for result in results:
    print(result)

In [ ]:
import sqlite3

if MAKE_DATABASE:
    conn = sqlite3.connect("biblia.db")
    c = conn.cursor()

    # Cria as tabelas
    c.execute(
        """
        CREATE TABLE IF NOT EXISTS versiculos (
            id INTEGER PRIMARY KEY,
            book TEXT,
            chapter INTEGER,
            verse INTEGER,
            text TEXT,
            line_number INTEGER
        )
    """
    )

    # Insere os dados
    for doc in docs:
        c.execute(
            """
            INSERT INTO versiculos (book, chapter, verse, text, line_number) VALUES (?, ?, ?, ?, ?)
        """,
            (
                doc.metadata["book"],
                doc.metadata["chapter"],
                doc.metadata["verse"],
                doc.page_content,
                doc.metadata["line_number"],
            ),
        )

    conn.commit()
    conn.close()